In [1]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.nn import Parameter
import torch.nn.functional as F
from torch.optim import Adam, SGD, AdamW
from torch.utils.data import DataLoader, Dataset
from torch.utils.checkpoint import checkpoint

#os.system('pip uninstall -y transformers')
#os.system('python -m pip install --no-index --find-links=../input/nbme-pip-wheels transformers')
import tokenizers
import transformers
print(f"tokenizers.__version__: {tokenizers.__version__}")
print(f"transformers.__version__: {transformers.__version__}")
from transformers import AutoTokenizer, AutoModel, AutoConfig
from transformers import get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup
from transformers import DataCollatorWithPadding

tokenizers.__version__: 0.21.0
transformers.__version__: 4.48.0


In [2]:
from dotenv import dotenv_values

CONFIG = dotenv_values(".env")

os.environ["SSL_CERT_FILE"] = "/Users/PChung2/.certificate/cacert.pem"
os.environ["REQUESTS_CA_BUNDLE"] = "/Users/PChung2/.certificate/cacert.pem"

In [3]:
text = [
    "The competition dataset comprises about 24000 student-written argumentative essays",
    "Each essay was scored on a scale of 1 to 6",
    "Your goal is to predict the score an essay received from its text."
]

In [31]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

In [42]:
input = tokenizer(  
    text,  
    max_length=128,  
    padding="max_length",    # Pad to 128 tokens  
    truncation=True,  
    return_tensors="pt"  
)

In [48]:
input['input_ids']

tensor([[  101,  1996,  2971,  2951, 13462,  8681,  2055, 11212,  8889,  3076,
          1011,  2517,  6685,  8082,  8927,   102,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,  

In [46]:
input['attention_mask'][0]

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])

In [37]:
tokenizer.convert_ids_to_tokens(1996)

'the'

In [ ]:
tokenizer.convert_ids_to_tokens(input['input_ids'][0])

['[CLS]',
 'the',
 'competition',
 'data',
 '##set',
 'comprises',
 'about',
 '240',
 '##00',
 'student',
 '-',
 'written',
 'argument',
 '##ative',
 'essays',
 '[SEP]']

In [49]:
len(input['input_ids'][0])

128

In [51]:
input['input_ids'].shape

torch.Size([3, 128])

In [64]:
input['attention_mask']

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [52]:
output = model(**input)

In [88]:
len(output)

2

In [89]:
output[0]

tensor([[[-4.3539e-01,  1.5306e-01,  1.0594e-01,  ..., -1.9136e-02,
           1.2085e-01,  6.5020e-01],
         [-9.4010e-01, -9.0715e-02, -1.7380e-01,  ...,  1.7172e-01,
           5.7382e-01,  1.1604e-01],
         [-6.7572e-01, -5.4717e-02,  1.1955e-01,  ..., -3.2578e-01,
          -5.2421e-02,  2.2422e-02],
         ...,
         [-2.8490e-01,  7.8289e-02,  3.8503e-01,  ...,  2.3455e-01,
           8.7386e-02,  1.5560e-01],
         [-1.9223e-01,  1.8693e-01,  3.5692e-01,  ...,  2.3348e-01,
           1.0926e-01,  2.1647e-01],
         [-4.5891e-01, -2.0612e-01,  3.1035e-01,  ...,  3.3568e-01,
           1.7520e-01,  1.4998e-01]],

        [[-2.1009e-01,  1.4119e-01,  2.0514e-01,  ..., -2.2084e-01,
           2.2582e-01,  6.1033e-01],
         [-9.3000e-01,  5.3456e-01, -2.2542e-01,  ...,  1.5978e-01,
           9.0411e-01,  2.1447e-01],
         [-5.3374e-01,  1.0148e-01, -5.6371e-01,  ...,  1.5970e-02,
           2.3357e-01, -2.6269e-01],
         ...,
         [-4.4680e-01, -2

In [90]:
output[1]

tensor([[-0.8588, -0.1946, -0.4374,  ..., -0.4187, -0.5653,  0.7178],
        [-0.8285, -0.1475,  0.1210,  ..., -0.1222, -0.5636,  0.7420],
        [-0.7957, -0.3769, -0.7542,  ..., -0.7734, -0.6643,  0.7663]],
       grad_fn=<TanhBackward0>)

In [59]:
input.keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])

In [55]:
output[0].shape

torch.Size([3, 128, 768])

In [60]:
class MeanPooling(nn.Module):
    def __init__(self):
        super(MeanPooling, self).__init__()
        
    def forward(self, last_hidden_state, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask
        return mean_embeddings

In [61]:
mp = MeanPooling()

In [66]:
mp_output = mp(output[0], input['attention_mask'])

In [67]:
mp_output.shape

torch.Size([3, 768])

In [75]:
output[0].shape

torch.Size([3, 128, 768])

In [72]:
cls_output = output[0][:, 0, :]

In [73]:
cls_output.shape

torch.Size([3, 768])

In [81]:
layer_norm = nn.LayerNorm(768)

In [82]:
mp_output = layer_norm(mp_output)

In [83]:
mp_output.shape

torch.Size([3, 768])

In [76]:
linear_layer = nn.Linear(768, 6)

In [84]:
mp_lin_output = linear_layer(mp_output)

In [85]:
mp_lin_output.shape

torch.Size([3, 6])

In [86]:
mp_lin_output

tensor([[-0.2541, -0.5642, -0.3351,  0.0343,  0.4244,  1.3433],
        [ 0.3512, -0.7683, -0.3034, -0.2815,  0.4742,  1.5240],
        [-0.0998, -0.4787, -0.5337, -0.0832,  0.6844,  1.6501]],
       grad_fn=<AddmmBackward0>)

In [79]:
cls_lin_output = linear_layer(cls_output)

In [80]:
cls_lin_output.shape

torch.Size([3, 6])

In [ ]:
(BS, 128) -> Into Base Model (BERT Base) -> (BS, 128, 768)

In [ ]:
(BS, 128, 768) OR (BS, CONTEXT_LEN, LAST_HIDDEN_DIM)

In [ ]:
(BS, 128 x 768) -> (128 x 768, 6)

In [ ]:
(BS, 768) -> (768, 6)